In [ ]:
# Demo

# Overview
- MNIST dataset
  + Goal: recognizing handwritten digits
  + Input: a 28x28 image
  + Output: class label (0, 1,..., 9)
- Defining an MLP for MNIST
- Training and evaluating an MLP

# Import libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
num_classes = 10   # 10 classes: 0, 1,..., 9

# (1) Data

In [ ]:
# the data, shuffled and split between train and test sets
train_set = datasets.MNIST(root='./data', train=True, download=True)
test_set = datasets.MNIST(root='./data', train=False, download=True)
x_train, y_train = train_set.data.numpy(), train_set.targets.numpy()
x_test, y_test = test_set.data.numpy(), test_set.targets.numpy()

x_train = x_train.reshape(60000, 28, 28)
x_test = x_test.reshape(10000, 28, 28)
x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
x_train /= 255
x_test /= 255
print(x_train.shape[0], 'train samples')
print(x_test.shape[0], 'test samples')

# PyTorch's nn.CrossEntropyLoss expects class indices (0..9), not one-hot vectors,
# so the labels are kept as integer class indices
y_train = y_train.astype('int64')
y_test = y_test.astype('int64')

# wrap numpy arrays into TensorDatasets (used by the DataLoaders below)
train_dataset = TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train))
test_dataset = TensorDataset(torch.from_numpy(x_test), torch.from_numpy(y_test))

# (2) Declare model

In [ ]:
def keras_init(m):
    # Keras default initialization: glorot_uniform weights, zero biases
    # (PyTorch's default init trains noticeably slower with the same SGD/Adam settings)
    if isinstance(m, (nn.Linear, nn.Conv2d)):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)


class MnistMLP(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # matrix (28x28 MNIST image) ---> vector
        self.flatten = nn.Flatten()

        # FC layers + dropout
        self.fc1 = nn.Linear(28 * 28, 512)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(p=0.2)

        self.fc2 = nn.Linear(512, 512)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(p=0.2)

        # output layer: returns raw logits (softmax is applied inside nn.CrossEntropyLoss,
        # or explicitly with F.softmax when we need probabilities)
        self.fc3_10ways_softmax = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = self.dropout1(self.relu1(self.fc1(x)))
        x = self.dropout2(self.relu2(self.fc2(x)))
        return self.fc3_10ways_softmax(x)


# define model
model = MnistMLP(num_classes=num_classes).to(device)
model.apply(keras_init)

# print model summary
print(model)
print('Total params:', sum(p.numel() for p in model.parameters()))
print('Trainable params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
# model structure (PyTorch has no built-in plot_model; print the module tree instead)
print(model)

# (3) Train defined model
- Note that the training history including loss and accuracy will be save in 'history' variable (a dict with keys 'loss', 'acc', 'val_loss', 'val_acc')
- In case your system runs out of memory (OOM), try to decrease batch size

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_correct += (logits.argmax(dim=1) == yb).sum().item()
        total += xb.size(0)
    return total_loss / total, total_correct / total


@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        total_loss += loss_fn(logits, yb).item() * xb.size(0)
        total_correct += (logits.argmax(dim=1) == yb).sum().item()
        total += xb.size(0)
    return total_loss / total, total_correct / total


def fit(model, train_loader, val_loader, epochs, loss_fn, optimizer):
    history = {'loss': [], 'acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(epochs):
        loss, acc = train_one_epoch(model, train_loader, loss_fn, optimizer)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn)
        history['loss'].append(loss)
        history['acc'].append(acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        print('Epoch %d/%d - loss: %.4f - acc: %.4f - val_loss: %.4f - val_acc: %.4f'
              % (epoch + 1, epochs, loss, acc, val_loss, val_acc))
    return history


@torch.no_grad()
def predict(model, x):
    """x: numpy array of shape (N, 28, 28) -> numpy array of class probabilities (N, 10)"""
    model.eval()
    logits = model(torch.as_tensor(x, dtype=torch.float32, device=device))
    return F.softmax(logits, dim=1).cpu().numpy()

In [ ]:
# declare learning rate, loss function, and optimizer
loss_fn = nn.CrossEntropyLoss()   # = softmax + categorical cross-entropy
lr = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=lr)

# train the model
batch_size = 128
epochs = 30

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

starting_time = time.time()
history = fit(model, train_loader, test_loader, epochs, loss_fn, optimizer)
print('> training time is %.4f minutes' % ((time.time() - starting_time)/60))

### Training time
- Training time with GTX 1080 is about 1.4402 minutes for 30 epochs (measured with the original Keras version)
- Training time with Core i7 is around 3.0493 minutes for 30 epochs (measured with the original Keras version)

# (4) Evaluate trained model

In [ ]:
score = evaluate(model, test_loader, loss_fn)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

In [ ]:
# select a test image randomly
random_test_index = np.random.choice(x_test.shape[0], size=1)[0]
test_img = x_test[random_test_index]
test_label = y_test[random_test_index]

# predict test image with trained model
pred_label = predict(model, np.expand_dims(test_img, axis=0))
pred_label = np.argmax(pred_label)

plt.imshow(test_img, cmap='gray')
plt.title('true label = %d, predicted label = %d' % (test_label, pred_label))
plt.show()

In [ ]:
random_test_index = np.random.choice(x_test.shape[0], size=1)[0]
test_img = x_test[random_test_index]
test_label = y_test[random_test_index]

In [ ]:
plt.imshow(test_img, cmap='gray')

In [ ]:
test_label

In [ ]:
test_img.shape

In [ ]:
# PyTorch modules do not store input/output shapes; check them with a dummy batch
dummy = torch.zeros(1, 28, 28, device=device)
model.eval()
tuple(dummy.shape), tuple(model(dummy).shape)

In [ ]:
np.expand_dims(test_img, axis = 0).shape

In [ ]:
y_pred = predict(model, np.expand_dims(test_img, axis = 0))

In [ ]:
y_pred.shape

In [ ]:
np.sum(y_pred, axis = 1)

In [ ]:
np.argmax(y_pred, axis = 1)

In [ ]:
# the 'fc2' layer (Linear + ReLU, like Keras Dense(activation='relu'))
model.fc2, model.relu2

In [ ]:
# extract the intermediate output of fc2 (after its ReLU) with a forward hook
features = {}

def save_fc2_output(module, inputs, output):
    features['fc2'] = output.detach()

hook_handle = model.relu2.register_forward_hook(save_fc2_output)

In [ ]:
y_pred = predict(model, np.expand_dims(test_img, axis = 0))
y_feat = features['fc2'].cpu().numpy()
hook_handle.remove()

In [ ]:
y_feat.shape, y_pred.shape

In [ ]:
y_feat